# Satellite Laser Ranging Station Data and Operational Anomaly Detection

<span style="color:blue">**Input required from user in next block:**</span>

In [ ]:
#select the station
padID = 7090
stationCode = "YARL"

"""
Provide the time period to be analyzed

Note on startYear and startMonth:
- a set number of days (numberOfDaysBack) is used to build the model and is included in the startYear and startMonth
- for example, given startYear = 2023, startMonth=3, numberOfDaysBack=60; the analysis will run for dates starting on 2023/03/01
- the length of time affects the runtime the most; a timespan of a year or less may be preferable
"""
startYear = 2024
startMonth = 1
endYear = 2024
endMonth = 9

"""
Monthly files may be downloaded from the CDDIS to create CSVs used to create the dataframes for the model
- Advantages = most up-to-date data including replacements will be used
- Disadvantage = runtime is long; best to limit to a few years at most

Note:
- it's possible to use your own file but you'll need to either need to create a new cell to read create the CSV
  or provide a filename that matches the year month format (the data within the file doesn't actually need to fit the timespan)
"""
#determine if monthly files should be downloaded
skipDownload = False
#determine if csv from monthly files need to be regenerated
skipCsv = False

"""
Provide the model parameters for the isolation forest - please note that these were not automated due to runtimes

Note:
- satelliteList:
   - standard values: LAGEOS and LARES (this software was originally created prior to LARES-2)
   - Note: to add to the satelliteList, use the names of satellite directories under: https://cddis.nasa.gov/archive/slr/data/npt_crd_v2/
- numberOfDaysBack: the number of days reviewed to create the model
   - standard value: 80-100 days
   - lower = higher accuracy
   - higher = greater number of NPT to review
   - Note: isolation forests prefer ~10,000 records per realization but this has been running with ~1,000 records
- checkDays: the number of days that are tested after the numberOfDaysBack for anomalies
   - standard value: set to 7 days for a weekly review
   - Note: minimum of 3 required because the an anomaly is only logged if it's found in 3/7 days
- contamination: variable within isolation forests that provides what percentage of the test data is expected to be anomalous
   - standard value: 0.008-0.05
   - lower = may miss important changes
   - higher = more false guesses

"""
satelliteList = ["lageos1", "lageos2", "lares", "lares2"]
numberOfDaysBack = 90
checkDays = 7
contamination = 0.05

In [ ]:
#imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
import plotly.express as px
import datetime
import requests
import sys
import logging

In [ ]:
#add logging
logger = logging.getLogger(__name__)
logging.basicConfig(
    filename = 'sample_stationAnomalyDetection.log',
    filemode = 'w',
    format   = ("%(asctime)s | %(levelname)s | %(funcName)s | %(lineno)s | %(message)s"),
    level    = logging.INFO
)

#log initial values from user
logger.info("Station " + str(padID) + " " + stationCode)
logger.info("Start Date: " + str(startYear) + "/" + str(startMonth))
logger.info("End Date: " + str(endYear) + "/" + str(endMonth))
logger.info("Skip Download: " + str(skipDownload))
logger.info("Skip CSV: " + str(skipCsv))
logger.info(satelliteList)

# Download NPT Data

In [ ]:
#method to download data
def dataDownload(satellite, year, month):
    if month < 10:
        strmonth = "0" + str(month)
    else:
        strmonth = str(month)
    if (year >= 2023 
        or (year == 2022 and month >= 8)):
        url = ("https://cddis.nasa.gov/archive/slr/data/npt_crd_v2/" 
            + satellite + "/" + str(year) + "/" + satellite + "_" + str(year) + strmonth + ".np2")
    else:
        url = ("https://cddis.nasa.gov/archive/slr/data/npt_crd/" 
            + satellite + "/" + str(year) + "/" + satellite + "_" + str(year) + strmonth + ".npt")
    logger.info(url)

    # Assigns the local file name to the last part of the URL
    filename = url.split('/')[-1]

    # Makes request of URL, stores response in variable r
    r = requests.get(url)
    if "H1" == r.text[:2].upper():
        logger.info("Correct download")
        # Opens a local file of same name as remote file for writing to
        with open("downloadedData/" + filename, 'wb') as fd:
            for chunk in r.iter_content(chunk_size=1000):
                fd.write(chunk)
        # Closes local file
        fd.close()
    else:
        logger.error("Please check the link - correct content could not be downloaded")

In [ ]:
#download data for satellites
if skipDownload == False:

    for satellite in satelliteList:
        for year in range(startYear, endYear+1):
            if startYear == endYear:
                for month in range(startMonth, endMonth+1):
                    dataDownload(satellite, year, month)
            else:
                if year == startYear:
                    for month in range(startMonth, 13):
                        dataDownload(satellite, year, month)
                elif year == endYear:
                    for month in range(1, endMonth+1):
                        dataDownload(satellite, year, month)
                else:
                    for month in range(1, 13):
                        dataDownload(satellite, year, month)

# Read NPT and Create CSV

<span style="color:blue">**Optional Changes from User: Add additional features for model**</span>

The code block below:
1. Reads the monthly files under downloadedData/
2. Extracts features for the model
3. Saves the features to CSVs under csvData/

Users have the option to add additional features from the CRD format.

In [ ]:
#read the downloaded files and create a CSV for easy data analysis
if skipCsv == False:
    #read files and write content to csv
    fileList = os.listdir("downloadedData/")
    for file0 in fileList:
        if file0[0] != ".":
            logger.info("Creating " + file0)
            wFile = open("csvData/" + file0.split(".")[0] + ".csv", "w")
            wFile.write("station,satellite,startTime,endTime," #obtained from header records
                        + "first11,last11,timeDiff," #obtained from the first and last 11 records
                        + "rmsCalc," #calculated from RMS in 11 records
                        + "rms40,systemDelay," #obtained from 40 records 
                        + "numberOfPoints," #number of 11 records with more then 5 raw ranges
                        + "rawRanges" #obtained from 11 record
                        + "\n"
                       )
            rFile = open("downloadedData/" + file0, "r", encoding="utf8")
            for line in rFile:
                recordType = line[:2].upper()
                #extract variables that will be used
                if recordType == "H1":
                    first11 = True
                    rawRanges = 0.0
                    rmsCalc = 0.0
                    numRMS = 0.0
                elif recordType == "H2":
                    station = line.split()[2]
                elif recordType == "H3":
                    satellite = line.split()[3]
                elif recordType == "H4":
                    lineVar = line.split()
                    startDT = (lineVar[2] + "/" 
                               + lineVar[3] + "/"
                               + lineVar[4] + " "
                               + lineVar[5] + ":"
                               + lineVar[6] + ":"
                               + lineVar[7]
                              )
                    endDT = (lineVar[8] + "/" 
                               + lineVar[9] + "/"
                               + lineVar[10] + " "
                               + lineVar[11] + ":"
                               + lineVar[12] + ":"
                               + lineVar[13]
                              )
                elif recordType == "11":
                    lineVar = line.split()
                    if len(lineVar) < 9:
                        continue
                    if first11 == True:
                        first11 = lineVar[1]
                    last11 = lineVar[1]
                    """
                    * The RMS of the pass is not always provided by the station so is calculated from the
                      ((speed of light in meters per second)/2)
                      *sum(RMS from the 11 records where the number of raw ranges > 5)
                      /(total number of 11 records where the number of raw ranges > 5)
                    * Note that the speed of light is divided by two because it's a one way range
                    """
                    rawRanges += float(lineVar[6])
                    if int(lineVar[6]) > 5:
                        rmsCalc += float(lineVar[7])
                        numRMS += 1
                elif recordType == "40":
                    lineVar = line.split()
                    systemDelay = str(float(lineVar[7])*0.299792458/2)
                    rms40 = str(float(lineVar[9])*0.149896229)
                elif recordType == "H8":
                    if first11 != True and numRMS > 0:
                        if first11 < last11:
                            wFile.write(station + ","
                                        + satellite + ","
                                        + startDT + ","
                                        + endDT + ","
                                        + str(first11) + ","
                                        + str(last11) + ","
                                        + str(float(last11)-float(first11)) + ","
                                        + str(rmsCalc/numRMS*0.149896229) + ","
                                        + rms40 + ","
                                        + systemDelay + ","
                                        + str(numRMS) + ","
                                        + str(rawRanges)
                                        + "\n"
                                       )
            rFile.close()
            wFile.close()

# Set up dataframes

In [ ]:
#read CSV and add the information to the mainDF
def addToDF(year,month,mainDF):
    if month < 10:
        month = "0" + str(month)

    for satellite in satelliteList:
        logger.info("Add " + satellite + "_" + str(year) + str(month) + ".csv to DF")
        df = pd.DataFrame(pd.read_csv("csvData/" + satellite + "_" + str(year) + str(month) + ".csv"))
        mainDF = pd.concat([mainDF,df],axis=0)

    return mainDF

In [ ]:
#Read the CSV into dataframe
mainDF =  pd.DataFrame()
for year in range(startYear, endYear+1):
    if startYear == endYear:
        for month in range(startMonth, endMonth+1):
            if month < 10:
                strmonth = "0" + str(month)
            else:
                strmonth = str(month)
            for satellite in satelliteList:
                if mainDF.empty:
                    logger.info("Add " + satellite + "_" + str(year) + str(strmonth) + ".csv to DF")
                    mainDF = pd.DataFrame(pd.read_csv("csvData/" + satellite + "_" + str(year) + strmonth + ".csv"))
                else:
                    logger.info("Add " + satellite + "_" + str(year) + str(strmonth) + ".csv to DF")
                    df = pd.DataFrame(pd.read_csv("csvData/" + satellite +"_" + str(year) + strmonth + ".csv"))
                    mainDF = pd.concat([mainDF,df],axis=0)
    else:
        if year == startYear:
            for month in range(startMonth, 13):
                if month < 10:
                    strmonth = "0" + str(month)
                else:
                    strmonth = str(month)
                for satellite in satelliteList:
                    if month == startMonth:
                        logger.info("Add " + satellite + "_" + str(year) + str(strmonth) + ".csv to DF")
                        mainDF = pd.DataFrame(pd.read_csv("csvData/" + satellite + "_" + str(year) + strmonth + ".csv"))
                    else:
                        logger.info("Add " + satellite + "_" + str(year) + str(strmonth) + ".csv to DF")
                        df = pd.DataFrame(pd.read_csv("csvData/" + satellite + "_" + str(year) + strmonth + ".csv"))
                        mainDF = pd.concat([mainDF,df],axis=0)
        elif year == endYear:
            for month in range(1, endMonth+1):
                mainDF = addToDF(year, month, mainDF)
        else:
            for month in range(1, 13):
                mainDF = addToDF(year, month, mainDF)

In [ ]:
#set the features and apply corrections.  Additional features based on median are added

mainDF['satellite'] = mainDF['satellite'].apply(str)
mainDF['timeDiff'] = mainDF['timeDiff'].astype(float)
mainDF['rmsCalc'] = mainDF['rmsCalc'].astype(float)
mainDF['startTime'] = pd.to_datetime(mainDF['startTime'])
mainDF['date'] = pd.to_datetime(mainDF['startTime']).dt.date
mainDF['medianRmsCalc'] = mainDF.groupby('date')['rmsCalc'].transform('median')
mainDF['medianRms40'] = mainDF.groupby('date')['rms40'].transform('median')

In [ ]:
#set up station dataframe
stationDF = mainDF.loc[mainDF.station == padID]
stationDF = stationDF.iloc[:, 1:]

In [ ]:
stationDF.head(5)

In [ ]:
stationDF.tail(5)

<span style="color:blue">**Review Average Number of NPT per Day**</span>

You may want to adjust the numberOfDaysBack based on the average number of NPT available for the timespan

In [ ]:
#divide total number of lines by number of distinct days
print ("Number of Passes: " + str(len(stationDF.index)))
print ("Number of Days with Tracking: " + str(len(pd.unique(stationDF['date']))))
avgPerDay = (len(stationDF.index))/(len(pd.unique(stationDF['date'])))
print ("Average Number of NPT Per Day: " + str(avgPerDay))

logger.info("Number of Passes: " + str(len(stationDF.index)))
logger.info("Number of Days with Tracking: " + str(len(pd.unique(stationDF['date']))))
avgPerDay = (len(stationDF.index))/(len(pd.unique(stationDF['date'])))
logger.info("Average Number of NPT Per Day: " + str(avgPerDay))

In [ ]:
startDate = pd.unique(stationDF['date']).min()
endDate = pd.unique(stationDF['date']).max()

logger.info("First Day of Tracking: " + str(startDate))
logger.info("Last Day of Tracking:  " + str(endDate))

# General Plotting

Plots of the features against the start time are plotted to provide a visual that can be later used to verify the anomaly dates.  Most anomaly dates will have a visual discontinuity within the data.  However, this is not always the case given that the anomaly date detected may be more discrete and across multiple features.

In [ ]:
#Display the data against start time
for value in stationDF.columns:
    fig = px.scatter(stationDF, x = 'startTime', y=value)
    fig.show()

# Isolation Model for All LAGEOS Features

The model below will provide dates where anomalies were detected using isolation forests.  When an anomaly is detected, SHAP plots are generated to show how the features were weighed when the anomaly was detected.  If less then 25 passes are available for testing over a 7-day period, the analysis is not run.  The minimum required number of passes for LAGEOS is 8 per week, however, the analysis is able to geneate more accurate results at 25 passes.

In [ ]:
import shap

#Ignore warnings.  This is because panda prefers subsetting by index
import warnings
warnings.filterwarnings('ignore')

anomalyDateCounts = {}
notEnoughDataDate = []
iterSDate = startDate
model = IsolationForest(
    n_estimators=numberOfDaysBack*2,
    max_samples=int(avgPerDay*numberOfDaysBack*.2), 
    contamination = float(contamination)
)

varsUsing = [#'date',
             #'startTime',
             'satellite', 
             #'timeDiff', 
             ##'rmsCalc',
             'rms40', 
             'systemDelay', 
             #'numberOfPoints', 
             #'rawRanges', 
             'medianRmsCalc', 
             'medianRms40']

df_partial = stationDF.loc[:, ['date','startTime'] + varsUsing]

df_partial.dropna(axis=0,inplace=True)


while iterSDate < endDate - datetime.timedelta(days=numberOfDaysBack+checkDays):
    df_train = df_partial.loc[(df_partial['date'] > iterSDate) & (df_partial['date'] <= iterSDate + datetime.timedelta(days=numberOfDaysBack))]
    df_test = df_partial.loc[(df_partial['date'] > iterSDate + datetime.timedelta(days=numberOfDaysBack)) & (df_partial['date'] <= iterSDate + datetime.timedelta(days=numberOfDaysBack+checkDays))]
    if len(df_test)<25:
        iterSDate = iterSDate + datetime.timedelta(days=1)
        print (str(iterSDate) + " less than 25 points")
        logger.info(str(iterSDate) + " less than 25 points")
        notEnoughDataDate.append(iterSDate)
        continue
    if len(df_train)<100:
        iterSDate = iterSDate + datetime.timedelta(days=1)
        print (str(iterSDate) + " less than 100 points to model")
        logger.info(str(iterSDate) + " less than 100 points to model")
        notEnoughDataDate.append(iterSDate)
        continue
    model.fit(df_train.loc[:,varsUsing])

    df_test['anomaly'] = model.predict(df_test.loc[:,varsUsing])
    df_train['anomaly'] = 0
    anomaly = df_test.loc[(df_test.anomaly==-1)]
    
    if len(pd.unique(anomaly['date'])) >= 3:
        logger.info(pd.unique(anomaly['date']))
        for var in varsUsing:
            fig = px.scatter(pd.concat([df_train, df_test]), x = 'startTime', y=var, color = "anomaly")
            fig.show()
        explainer = shap.TreeExplainer(
            model, 
            masker=df_train.loc[:,varsUsing])
        shapvs = explainer(df_train.loc[:,varsUsing])
        shap.plots.bar(shapvs[0])#, feature_names=varsUsing)
        
        #shapvs = explainer.shap_values(
        #    df_train.loc[:,varsUsing], 
        #    check_additivity=True)
        #shap.summary_plot(shapvs, feature_names=varsUsing]
        for date in anomaly['date']:
            logger.info("Anomaly Detected on " + str(date))
            if date not in anomalyDateCounts:
                anomalyDateCounts[date] = 1
            else:
                anomalyDateCounts[date] += 1
    iterSDate = iterSDate + datetime.timedelta(days=1)


## Anomaly Dates Detected

For the anomaly dates detected, you'll want to look for clustering where the number of anomaly counts are higher which reflects a higher level of certainty

In [ ]:
anomalyDatesPlotVal = anomalyDateCounts.copy()
anomalyDatesPlotVal[startDate] = 1
anomalyDatesPlotVal[endDate] = 1

fig = px.scatter(x=anomalyDatesPlotVal.keys(), 
                 y=[x for x in list(anomalyDatesPlotVal.values())],
                )
fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Anomaly Counts")
fig.show()

In [ ]:
anomalyGuessList = []

for key in anomalyDateCounts.keys():
    anomalyGuessList.append(key)

for date in anomalyGuessList:
    print (date)
    logger.info(anomalyDateCounts)

# Compare Results Against Station Log File

To determine the accuracy of the model, the anomaly dates detected are compared against change dates available in the station log file where the influence is > 1 and the subsystem is 5 or 6.  Based on these outputs the model or model parameters can be adjusted.

In [ ]:
#Download Station History Log
url = "https://cddis.nasa.gov/archive/slr/slrhst/"

#Makes request of URL, stores response in variable r
r = requests.get(url + "*?list")

#Prints the results of the directory listing
for line in r.text.split("\n"):
    if line[:4] == stationCode.lower():
        stationFilename = (line.split()[0])
        logger.info("Station History Log: " + stationFilename)
        break

# Opens a local file of same name as remote file for writing to
stationFile = requests.get(url + stationFilename)
print (url + stationFilename)
logger.info(stationFile)

with open("stationHstLog.txt", 'wb') as fd:
    for chunk in stationFile.iter_content(chunk_size=1000):
        fd.write(chunk)
# Closes local file
fd.close()

In [ ]:
#Read Station History Log
logDateDict = {}
rFile = open("stationHstLog.txt", "r", encoding="utf8")
for line in rFile:
    if line[:4] == str(padID):
        lineVar = line.split()
        if (int(lineVar[5].split(".")[0]) in [5,6]):
            staYear = lineVar[1]
            staDoy = lineVar[2]
            staDate = datetime.datetime.strptime(staYear + "-" + staDoy, "%Y-%j")
            #if (staDate.date() >= pd.unique(stationDF['date']).min()):
            if (staDate.date() >= startDate + datetime.timedelta(days=numberOfDaysBack)
                and staDate.date() <= endDate):
                if staDate not in logDateDict:
                    logDateDict[staDate] = {"influence" : lineVar[4],
                                                "subsystem" : lineVar[5],
                                                "fullInfo" : line
                                               }
                else:
                    if logDateDict[staDate]["subsystem"] < lineVar[5]:
                        logDateDict[staDate] = {"influence" : lineVar[4],
                                                    "subsystem" : lineVar[5],
                                                    "fullInfo" : line
                                                   }
rFile.close()
for date in logDateDict:
    print (date)
    print (logDateDict[date])

In [ ]:
#Check Results Against Expected Dates
guessRemaining = anomalyGuessList.copy()
logRemaining = logDateDict.copy()

for logDate in logDateDict:
    print ("log: " + str(logDate))
    for guessDate in anomalyGuessList:
        #here we adjust in case the station personnel forgot to add the change date and are late by a week
        if (logDate.date() >= (guessDate - datetime.timedelta(days=14)) 
            and logDate.date() <= guessDate + datetime.timedelta(days=14)):
            print ("Found: log " + str(logDate) + " in detection " + str(guessDate))
            if logDate in logRemaining:
                del logRemaining[logDate]
            if guessDate in guessRemaining:
                guessRemaining.remove(guessDate)
            continue

for guessDate in anomalyGuessList:
    print ("Guess: " + str(guessDate))
    for logDate in logDateDict:
        #here we adjust in case the station personnel forgot to add the change date and are late by a week
        #we also allow for errors to drag for 2 weeks before becoming a part of the "New Normal"
        if (logDate.date() >= (guessDate - datetime.timedelta(days=14)) 
            and logDate.date() <= guessDate + datetime.timedelta(days=14)):
            print ("Found: detection " + str(guessDate) + " in log " + str(logDate.date()))
            if logDate in logRemaining:
                del logRemaining[logDate]
            if guessDate in guessRemaining:
                guessRemaining.remove(guessDate)
            continue
                
print ("\n\nFalse Alarms:")
for guess in guessRemaining:
    print (guess)

print ("\n\nFound in History Log, Not Detected: ")
for logDate in logRemaining:
    skipped = False
    if int(logRemaining[logDate]["influence"]) > 1:
        for noDataDay in notEnoughDataDate:
            if (logDate.date() >= noDataDay - datetime.timedelta(days=7)
               and logDate.date() <= noDataDay):
                skipped = True
                print (str(logDate) + " Not Detect, Not Enough Data") 
                break
        if skipped == False:
            print (logRemaining[logDate])